# Rotated Irap surface -> north-up GeoTIFF (NumPy only)

Pure NumPy + rasterio, no xtgeo. Reads a large Irap binary surface, resamples
the rotated grid onto an axis-aligned grid (nearest neighbour), and writes a
north-up GeoTIFF. Stays in float32 throughout.

In [ ]:
# Rotates (if needed) and converts Irap binary grids to GeoTIFF using only NumPy and Rasterio

import mmap

import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.transform import Affine

UNDEF = np.float32(1.0e30)  # Irap binary undefined value

In [ ]:
directory = '../rasters'
surface = 'ISOntFangstUJUNC.irb'
insurface = directory + '/' + surface

out_tif = '../rasters/ISOntFangstUJUNC_unrotated.tif'
epsg = 23032  # NB: set to the surface's real CRS (assumed ED50 / UTM zone 32N)
print(insurface)

../rasters/ISOntFangstUJUNC.irb


In [3]:
def read_irap_binary(path):
    """Read an Irap binary surface with pure NumPy.

    Safe for files >2GB on Windows/NumPy 2.x (byte offset kept a Python int).
    Returns a dict with geometry and a float32 values array of shape (ncol, nrow),
    where undefined nodes are NaN.
    """
    with open(path, 'rb') as fh:
        buf = mmap.mmap(fh.fileno(), 0, access=mmap.ACCESS_READ)

    # Header: <32> IDFLAG NY XORI XMAX YORI YMAX XINC YINC <32>
    #         <16> NX ROT X0ORI Y0ORI <16>  <28> 0*7 <28>
    hed = np.frombuffer(
        buf[:100],
        dtype='>i4,>i4,>i4,>f4,>f4,>f4,>f4,>f4,>f4,>i4,'
        '>i4,>i4,>f4,>f4,>f4,>i4,'
        '>i4,>i4,>i4,>i4,>i4,>i4,>i4,>i4,>i4',
    )
    nrow = int(hed[0][2])
    xori = float(hed[0][3])
    yori = float(hed[0][5])
    xinc = float(hed[0][7])
    yinc = float(hed[0][8])
    ncol = int(hed[0][11])
    rotation = float(hed[0][12])

    yflip = 1
    if yinc < 0.0:
        yinc = -yinc
        yflip = -1

    stv = 100  # Python int offset, never cast to numpy int
    datav = []
    nbytes = len(buf)
    while stv < nbytes:
        blockv = int(np.frombuffer(buf[stv:stv + 4], dtype='>i4')[0])
        stv += 4
        datav.append(np.frombuffer(buf[stv:stv + blockv], dtype='>f4'))
        stv += blockv + 4

    values = np.hstack(datav).astype(np.float32).reshape((ncol, nrow), order='F')
    values = np.ascontiguousarray(values)
    values[values >= 1.0e29] = np.nan
    del buf

    return {
        'ncol': ncol, 'nrow': nrow,
        'xori': xori, 'yori': yori,
        'xinc': xinc, 'yinc': yinc,
        'rotation': rotation, 'yflip': yflip,
        'values': values,
    }

In [4]:
surf = read_irap_binary(insurface)
for k, v in surf.items():
    if k != 'values':
        print(f'{k:>9}: {v}')
print('   values:', surf['values'].shape, surf['values'].dtype)

     ncol: 34156
     nrow: 17543
     xori: 295621.53125
     yori: 6715466.5
     xinc: 25.0
     yinc: 25.0
 rotation: 63.0
    yflip: 1
   values: (34156, 17543) float32


In [5]:
# Axis-aligned bounding box of the rotated surface (from its 4 corner nodes)
th = np.radians(surf['rotation'])
cos, sin = np.cos(th), np.sin(th)

ii = np.array([0, surf['ncol'] - 1, 0, surf['ncol'] - 1], dtype=np.float64)
jj = np.array([0, 0, surf['nrow'] - 1, surf['nrow'] - 1], dtype=np.float64)
cx = surf['xori'] + ii * surf['xinc'] * cos - jj * surf['yinc'] * sin
cy = surf['yori'] + ii * surf['xinc'] * sin + jj * surf['yinc'] * cos

xmin, xmax = cx.min(), cx.max()
ymin, ymax = cy.min(), cy.max()

ncol_out = int(np.ceil((xmax - xmin) / surf['xinc'])) + 1
nrow_out = int(np.ceil((ymax - ymin) / surf['yinc'])) + 1
print('output grid:', ncol_out, 'x', nrow_out, '=', ncol_out * nrow_out, 'cells')

output grid: 31138 x 38398 = 1195636924 cells


In [6]:
def resample_nearest(surf, xmin, ymin, ncol_out, nrow_out, nodata=UNDEF):
    """Nearest-neighbour resample of a rotated surface onto an axis-aligned grid.

    Returns a float32 array shape (ncol_out, nrow_out); nodata where empty.
    """
    src = surf['values']
    ncol, nrow = src.shape
    xinc, yinc = surf['xinc'], surf['yinc']
    th = np.radians(surf['rotation'])
    cos, sin = np.cos(th), np.sin(th)

    out = np.full((ncol_out, nrow_out), nodata, dtype=np.float32)
    dx = (xmin + np.arange(ncol_out, dtype=np.float64) * xinc) - surf['xori']

    for q in range(nrow_out):  # iterate output rows (Y)
        dy = (ymin + q * yinc) - surf['yori']
        i = np.rint((dx * cos + dy * sin) / xinc).astype(np.int64)
        j = np.rint((-dx * sin + dy * cos) / yinc).astype(np.int64)
        ok = (i >= 0) & (i < ncol) & (j >= 0) & (j < nrow)
        vals = src[i[ok], j[ok]]
        good = ~np.isnan(vals)
        cols = np.nonzero(ok)[0][good]
        out[cols, q] = vals[good]
    return out


unrotated = resample_nearest(surf, xmin, ymin, ncol_out, nrow_out)
print('resampled:', unrotated.shape, unrotated.dtype,
      'valid:', int(np.count_nonzero(unrotated < 1.0e29)))

resampled: (31138, 38398) float32 valid: 73824701


In [7]:
# (ncol, nrow) row-northward -> (nrow, ncol) north-up (row 0 = max Y)
arr = unrotated.T[::-1, :]
nrow_t, ncol_t = arr.shape

# Node positions are cell centres; shift half a cell to the pixel corner.
top_left_x = xmin - 0.5 * surf['xinc']
top_left_y = ymin + (nrow_out - 1) * surf['yinc'] + 0.5 * surf['yinc']

transform = Affine(surf['xinc'], 0.0, top_left_x,
                   0.0, -surf['yinc'], top_left_y)

with rasterio.open(
    out_tif, 'w', driver='GTiff',
    height=nrow_t, width=ncol_t, count=1, dtype=arr.dtype,
    crs=CRS.from_epsg(epsg), transform=transform, nodata=float(UNDEF),
    compress='LZW', tiled=True, bigtiff='IF_SAFER',
) as dst:
    dst.write(arr, 1)

print('Wrote', out_tif)

RasterioIOError: Attempt to create new tiff file 'rasters/ISOntFangstUJUNC_unrotated_numpy.tif' failed: rasters/ISOntFangstUJUNC_unrotated_numpy.tif: No such file or directory